## Single-Task Learning (STL) vs. Multitask Learning (MTL) in Feed-Forward Neural Networks (FNN)

### 1. Model Setup

Let  
- $X \in \mathbb{R}^d$: input feature vector  
- $f_\theta : \mathbb{R}^d \to \mathbb{R}^k$: FNN with parameters $\theta$

Shared hidden layers:
$$
h_1 = \sigma(W_1 X + b_1), \qquad
h_2 = \sigma(W_2 h_1 + b_2), \quad \dots
$$

---

## 2. Single-Task Learning (STL)

### Output
The model predicts one scalar:
$$
\hat{y} = f_\theta(X) \in \mathbb{R}.
$$

### Loss Function
Given dataset $\{(X_i, y_i)\}_{i=1}^n$:
$$
L(\theta) = \frac{1}{n} \sum_{i=1}^n \ell(f_\theta(X_i),\, y_i).
$$

Squared error:
$$
\ell(\hat{y}, y) = (\hat{y}-y)^2.
$$

### Gradient Update
$$
\theta \leftarrow \theta - \eta\, \nabla_\theta L(\theta).
$$

---

## 3. Multitask Learning (MTL)

Assume $T$ related tasks.

### Outputs
$$
\hat{y} = f_\theta(X) =
\begin{pmatrix}
\hat{y}^{(1)} \\
\hat{y}^{(2)} \\
\vdots \\
\hat{y}^{(T)}
\end{pmatrix}.
$$

Each task has its own output head:
$$
\hat{y}^{(t)} = W^{(t)} h_L + b^{(t)}.
$$

### Loss Function
Task-specific loss:
$$
L_t(\theta) = \frac{1}{n} \sum_{i=1}^n \ell_t(\hat{y}^{(t)}_i,\, y^{(t)}_i).
$$

Total loss:
$$
L(\theta) = \sum_{t=1}^T w_t\, L_t(\theta), \qquad w_t>0.
$$

### Gradient Update

Shared layers:
$$
\nabla_{\theta_{\text{shared}}} L 
= \sum_{t=1}^T w_t\, \nabla_{\theta_{\text{shared}}} L_t.
$$

Task heads:
$$
\nabla_{\theta^{(t)}_{\text{head}}} L 
= w_t\, \nabla_{\theta^{(t)}_{\text{head}}} L_t.
$$

---

## 4. Core Mathematical Difference

### STL Optimization
$$
\theta_{\text{STL}}^\star = \min_\theta L_1(\theta).
$$

### MTL Optimization
$$
\theta_{\text{MTL}}^\star = \min_\theta \sum_{t=1}^T w_t L_t(\theta).
$$

---

## 5. Gradient Comparison

### STL
$$
\nabla_\theta L_{\text{STL}} = \nabla_\theta L_1.
$$

### MTL
$$
\nabla_\theta L_{\text{MTL}} = \sum_{t=1}^T w_t\, \nabla_\theta L_t.
$$

---

## 6. Summary Table

| Aspect | STL | MTL |
|-------|-----|-----|
| Output | $f_\theta(X)\\in\\mathbb{R}$ | $f_\theta(X)\\in\\mathbb{R}^T$ |
| Objective | $L_1(\theta)$ | $\\sum_t w_t L_t(\theta)$ |
| Gradient | $\\nabla L_1$ | $\\sum_t w_t \\nabla L_t$ |
| Representation | Single task | Shared representation |
| Output heads | One | One per task |
| Learning signal | One task only | All tasks jointly |


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

In [ ]:
# Download 5 years of daily data
tickers = ["GLD", "^GSPC", "AAPL"]  # Gold ETF, S&P 500 index, Apple
data = yf.download(tickers, period="5y", interval="1d")["Adj Close"]
data = data.dropna()
asset_names = data.columns.tolist()
print(data.head())

In [ ]:
# Create sliding windows
WINDOW = 30

prices = data.values
dates  = data.index
N, A   = prices.shape

X = []
y = []
lp = []       # last-day price in window
t_dates = []  # next-day date

for i in range(WINDOW, N):
    X.append(prices[i-WINDOW:i].reshape(-1))   # flatten 30×3 → 90
    y.append(prices[i])
    lp.append(prices[i-1])
    t_dates.append(dates[i])

X = np.array(X)
y = np.array(y)
lp = np.array(lp)
t_dates = np.array(t_dates)

print("X shape:", X.shape)
print("y shape:", y.shape)

<b> Include all the features you want to use, here I include the validation dataset, you can choose not to use it. Just need to remember that the datasets used for all model should be same. </b>

In [ ]:
# Train/Val/Test split
n_total = len(X)
n_train = int(0.7 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

X_train = X[:n_train]
y_train = y[:n_train]

X_val   = X[n_train:n_train+n_val]
y_val   = y[n_train:n_train+n_val]

X_test  = X[n_train+n_val:]
y_test  = y[n_train+n_val:]
lp_test = lp[n_train+n_val:]
dates_test = t_dates[n_train+n_val:]

print("Train:", X_train.shape[0])
print("Val:", X_val.shape[0])
print("Test:", X_test.shape[0])

In [ ]:
# Scale features & targets
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train)
X_val_s   = x_scaler.transform(X_val)
X_test_s  = x_scaler.transform(X_test)

y_train_s = y_scaler.fit_transform(y_train)
y_val_s   = y_scaler.transform(y_val)

<b> Make sure you make the tuning grids for STL and MTL are the same </b>

In [ ]:
# Hard-coded basic hyperparameter tuning
hidden_choices = [32, 64]
lr_choices = [1e-3, 5e-4]

best_val = np.inf
best_h = None
best_lr = None

for h in hidden_choices:
    for lr in lr_choices:
        
        model = Sequential([
            Dense(h, activation='relu', input_shape=(X_train_s.shape[1],)),
            Dense(h//2, activation='relu'),
            Dense(3)   # GLD, SP500, AAPL
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                      loss='mse')

        print(f"Training: units={h}, lr={lr}")
        hist = model.fit(
            X_train_s, y_train_s,
            validation_data=(X_val_s, y_val_s),
            epochs=15,
            batch_size=32,
            verbose=0
        )

        val_mse = min(hist.history["val_loss"])
        print("  val MSE =", val_mse)

        if val_mse < best_val:
            best_val = val_mse
            best_h = h
            best_lr = lr

print("\nBest hyperparameters:", best_h, best_lr)

In [ ]:
# Train final model (train+val)
X_full = np.vstack([X_train_s, X_val_s])
y_full = np.vstack([y_train_s, y_val_s])

final_model = Sequential([
    Dense(best_h, activation='relu', input_shape=(X_train_s.shape[1],)),
    Dense(best_h//2, activation='relu'),
    Dense(3)
])
final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=best_lr),
                    loss='mse')

final_model.fit(X_full, y_full, epochs=20, batch_size=32, verbose=1)

In [ ]:
# Predict (convert back to price)
y_pred_s = final_model.predict(X_test_s)
y_pred = y_scaler.inverse_transform(y_pred_s)   # convert to price
# Compute returns
actual_ret = (y_test / lp_test) - 1
pred_ret   = (y_pred / lp_test) - 1

In [ ]:
# Print metrics
for j, name in enumerate(asset_names):
    mse = mean_squared_error(actual_ret[:, j], pred_ret[:, j])
    mae = mean_absolute_error(actual_ret[:, j], pred_ret[:, j])
    rmse = np.sqrt(mse)

    print(f"\n{name}:")
    print("  MSE  =", mse)
    print("  MAE  =", mae)
    print("  RMSE =", rmse)

In [ ]:
# Plot actual vs predicted returns
plt.figure(figsize=(12, 9))

for j, name in enumerate(asset_names):
    plt.subplot(3, 1, j+1)
    plt.plot(dates_test, actual_ret[:, j], label="Actual", linewidth=1)
    plt.plot(dates_test, pred_ret[:, j], label="Predicted", alpha=0.7)
    plt.title(f"{name} - Actual vs Predicted Returns")
    plt.grid(True)
    if j == 0:
        plt.legend()

plt.tight_layout()
plt.show()